## 2.1 卷积层理论计算

**已知条件：**
- 输入图像尺寸：3 × 32 × 32（通道数 × 高 × 宽）
- 卷积核：16个，每个尺寸为 3 × 5 × 5
- Padding：2
- Stride：2

### 问题1：输出特征图尺寸

输出通道数 = 16

输出高度计算：
$$H_{out} = \left\lfloor \frac{H_{in} + 2P - H_k}{S} \right\rfloor + 1 = \left\lfloor \frac{32 + 4 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16$$

输出宽度计算同理：$W_{out} = 16$

**答案：** $16 \times 16 \times 16$（通道数 × 高 × 宽）

### 问题2：单个输出像素的乘法次数

单个卷积核参数量 = $3 \times 5 \times 5 = 75$

每个输出像素对应一次卷积操作，需要将所有卷积核权重与输入像素相乘

**答案：** 75 次乘法


In [1]:
import numpy as np

def max_pool2d_forward(x, kernel_size, stride=1, padding=0):
    """
    手动实现二维最大池化前向传播
    x: 输入张量，形状 (batch, channels, height, width)
    kernel_size: 池化窗口大小 (h, w) 或 int
    stride: 步幅 (h, w) 或 int
    padding: 填充大小 (h, w) 或 int
    """
    # 统一参数格式
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    if isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding

    batch, c, h, w = x.shape

    # 填充
    x_pad = np.pad(x, ((0,0), (0,0), (ph, ph), (pw, pw)), mode='constant')

    # 输出尺寸
    out_h = (h + 2*ph - kh) // sh + 1
    out_w = (w + 2*pw - kw) // sw + 1

    out = np.zeros((batch, c, out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            h_start = i * sh
            h_end = h_start + kh
            w_start = j * sw
            w_end = w_start + kw
            window = x_pad[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = np.max(window, axis=(2, 3))

    return out

## 3.1 VGG卷积层参数量计算

**已知条件：** 输入和输出通道数均为 $C$，无偏置

### 问题1：单个 $5 \times 5$ 卷积层参数量

$$参数量 = C \times C \times 5 \times 5 = 25C^2$$

### 问题2：两个串联 $3 \times 3$ 卷积层总参数量

第一层：$C \times C \times 3 \times 3 = 9C^2$  
第二层：$C \times C \times 3 \times 3 = 9C^2$  
总参数量：$9C^2 + 9C^2 = 18C^2$

**答案：** $25C^2$ 和 $18C^2$

In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)

## 4.1 批量归一化计算

**已知条件：**
- 输入值：$x_1 = 2, x_2 = 4, x_3 = 6, x_4 = 8$
- $\gamma = 2$，$\beta = 1$，$\epsilon = 0$

### 计算过程：

均值：
$$\mu = \frac{2+4+6+8}{4} = 5$$

方差：
$$\sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} = \frac{9+1+1+9}{4} = 5$$

归一化并变换：
$$y_i = \gamma \cdot \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

具体计算：
- $y_1 = 2 \cdot \frac{2-5}{\sqrt{5}} + 1 = 2 \cdot \frac{-3}{\sqrt{5}} + 1 = 1 - \frac{6}{\sqrt{5}}$
- $y_2 = 2 \cdot \frac{4-5}{\sqrt{5}} + 1 = 2 \cdot \frac{-1}{\sqrt{5}} + 1 = 1 - \frac{2}{\sqrt{5}}$
- $y_3 = 2 \cdot \frac{6-5}{\sqrt{5}} + 1 = 2 \cdot \frac{1}{\sqrt{5}} + 1 = 1 + \frac{2}{\sqrt{5}}$
- $y_4 = 2 \cdot \frac{8-5}{\sqrt{5}} + 1 = 2 \cdot \frac{3}{\sqrt{5}} + 1 = 1 + \frac{6}{\sqrt{5}}$

**答案：** $y_1 = 1 - \frac{6}{\sqrt{5}}, y_2 = 1 - \frac{2}{\sqrt{5}}, y_3 = 1 + \frac{2}{\sqrt{5}}, y_4 = 1 + \frac{6}{\sqrt{5}}$

In [3]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.use_1x1conv = use_1x1conv
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.conv3 = None

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.conv3:
            x = self.conv3(x)
        return torch.relu(out + x)

## 5.1 微调理论问题

### 问题1：学习率设置策略

**答案：** 底层特征提取层在源数据集（如ImageNet）上已经学到了通用的特征（边缘、纹理等），这些特征具有较好的泛化性，因此应使用较小的学习率或将其冻结，以保留已学到的知识。而顶层输出层需要适应新的目标任务，因此使用较大的学习率来快速学习任务特定的特征。

### 问题2：小数据集微调策略

**答案：** 
- 冻结大部分底层特征提取层，只微调最后几层
- 使用较小的学习率，避免过拟合
- 增强正则化（如增大dropout率、权重衰减）
- 增加数据增广来提高泛化能力

In [4]:
from torchvision import transforms

augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

## 6.1 IoU计算

**已知条件：**
- 真实框 A = [10, 10, 50, 50]
- 预测框 B = [30, 30, 70, 70]

### 计算过程：

交集坐标：
- 左上角 x：$\max(10, 30) = 30$
- 左上角 y：$\max(10, 30) = 30$
- 右下角 x：$\min(50, 70) = 50$
- 右下角 y：$\min(50, 70) = 50$

交集面积 = $(50-30) \times (50-30) = 20 \times 20 = 400$

A框面积 = $(50-10) \times (50-10) = 40 \times 40 = 1600$

B框面积 = $(70-30) \times (70-30) = 40 \times 40 = 1600$

并集面积 = $1600 + 1600 - 400 = 2800$

$$IoU = \frac{400}{2800} = \frac{1}{7}$$

**答案：** $\frac{1}{7}$（约等于 0.1429）

In [5]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1):
    """
    logits: 模型输出 (batch, num_classes)
    labels: 真实标签 (batch,)
    epsilon: 平滑因子
    """
    num_classes = logits.shape[-1]
    log_probs = F.log_softmax(logits, dim=-1)

    # 构建平滑标签
    smooth_labels = torch.full_like(log_probs, epsilon / (num_classes - 1))
    smooth_labels.scatter_(1, labels.unsqueeze(1), 1.0 - epsilon)

    # 计算损失
    loss = - (smooth_labels * log_probs).sum(dim=-1).mean()
    return loss